In [ ]:
from pathlib import Path

import numpy
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tqdm import tqdm

In [ ]:
#strainGE_dir= Path("../Respiratory/StrainGE/")
strainGE_dir= Path("../Respiratory/MAGs_StrainGE/")

In [ ]:
straingst_df = pd.read_table(strainGE_dir/"StrainGST.tsv", index_col=[0,1])
straingst_df.head()

In [ ]:
STRAINGR_DIR = strainGE_dir/"Intermediate_files/StrainGR/summary/"

df_list = []
sample_names = []
for f in STRAINGR_DIR.glob("*.tsv"):
    df = pd.read_csv(f, sep='\t', index_col=0)
    df = df.drop(index='TOTAL')  # Remove TOTAL statistics

    df_list.append(df)
    sample_names.append(f.stem)

straingr_df = pd.concat(df_list, keys=sample_names, names=["sample"])

straingr_df['straingst_present'] = straingr_df.index.isin(straingst_df.index)

straingr_df.to_csv(strainGE_dir /"StrainGR_summary.tsv", sep='\t')

In [ ]:
straingr_df.head()

In [ ]:
# aggregate

straingr_df.eval("Density= coverage*length",inplace=True)
straingr_agg = straingr_df.groupby(level=[0,1]).sum()



#drop percentage
straingr_agg= straingr_agg.loc[:,~ straingr_agg.columns.str.contains("Pct")]

straingr_agg.drop(["median","abundance","coverage"],axis=1,inplace=True)

# length weighted coverage 
straingr_agg.eval("coverage = Density/length",inplace=True)

straingr_df = straingr_agg


In [ ]:

straingr_df['is_plasmid'] = straingr_df['length'] < 4e4
straingr_df['enough_cov'] = straingr_df['coverage'] > 0.5

straingr_df['straingst_present'] = straingr_df.index.isin(straingst_df.index)

In [ ]:


# Filter and re-index
straingr_df = straingr_df[straingr_df['straingst_present'] & ~straingr_df['is_plasmid'] & straingr_df['enough_cov']].reset_index().set_index(['sample', 'ref'])
straingr_df

In [ ]:
straingr_df.groupby(level=1).size()

In [ ]:
straingr_df.groupby(level=[0,1]).size().unstack().to_csv( strainGE_dir/"Strain_called.tsv",sep='\t')

# Comparison data

not yet finished

In [ ]:
Comparison_dir= strainGE_dir/"Intermediate_files/StrainGR/comparisons/summary/"

# increases with time
comparison_list = list(Comparison_dir.glob("*.tsv"))
print(len(comparison_list) )

df_list =[]
for file in tqdm(comparison_list):
    try: 
        df= pd.read_table(file, index_col=[0,1,2])
        df_list.append(df)
    except pd.errors.EmptyDataError:
        pass
    
compare_df = pd.concat(df_list)
compare_df.to_csv(strainGE_dir/"Comparisons.tsv",sep="\t")

# Comparison

In [ ]:

compare_df = pd.read_csv(strainGE_dir /"Comparisons.tsv", sep='\t', index_col=[0, 1, 2])

In [ ]:
#aggregate

compare_agg = compare_df.groupby(level=[0,1,2]).sum()
compare_agg.eval("singleAgreePct = singleAgree/single*100", inplace=True)
compare_agg.eval("commonPct = common/length*100", inplace=True)


compare_df= compare_agg

In [ ]:
compare_agg.columns

In [ ]:
# comparisoncolumns
unique_sample_genome = straingr_df.index.unique()

def both_straingst_present(ix):
    sample1, sample2, ref = ix

    return (sample1, ref) in unique_sample_genome and (sample2, ref) in unique_sample_genome

compare_df['both_present'] = compare_df.index.map(both_straingst_present)
compare_df = compare_df[compare_df['both_present']].copy()


# add species and genus

compare_df["Species"] = compare_df.index.get_level_values(2).map(lambda s: " ".join(s.split("_",2)[:2]))
compare_df["Genus"] = compare_df.index.get_level_values(2).str.split("_",1,expand=True).to_frame()[0].map({"Stre":"Streptococcus","Stap":"Staphylococcus"}).values




all_compare_df= compare_df.copy()


In [ ]:
compare_df.head()

In [ ]:
# filter
compare_df= all_compare_df.query("commonPct>20")

In [ ]:
#plot

import seaborn as sns

sns.scatterplot(x="gapJaccardSim", y="singleAgreePct", size="commonPct", data=compare_df,hue="ref")#.query("Genus=='Streptococcus'"), hue="Species")

#plt.xlim(0.970, 1)
plt.xlabel("Gap similarity")

#plt.ylim(99.9, 100)
plt.ylabel("ACNI")
plt.gca().yaxis.set_major_formatter("{x:g}%")

plt.grid('on')
plt.legend(title="Common\nCallable [%]", loc="center left", bbox_to_anchor=(1.05, 0.5))

In [ ]:
#plot

import seaborn as sns

sns.distplot(x="ref", y="singleAgreePct", data=compare_df.reset_index())#.query("Genus=='Streptococcus'"), hue="Species")

#plt.xlim(0.970, 1)


#plt.ylim(99.9, 100)
plt.ylabel("ACNI")


plt.grid('on')


StrainGR compares positions within the ?callable genome? or the set of positions with any reference or alternative allele supported by at least two good reads and?>?10% of the alignment pileup (Fig. 1c). To perform a comparison, only ?common callable? positions are considered, which represent the subset of the callable genome for a given reference that is shared by two samples. Strain relationships can be assessed using two key metrics: (i) the Average Callable Nucleotide Identity (ACNI), or the percentage of common callable positions where both samples have a single identical base call, and (ii) a ?gap similarity? metric, as patterns of large deletions are often conserved between closely related strains, which can provide an orthogonal metric of strain similarity [23]. The ACNI and gap similarity values that define two samples as containing the same ?strain? depend on the research question [7]. For the purposes of this manuscript, we consider two samples to contain the same strain if ACNI is ??99.95%, which

In [ ]:
compare_df.query("singleAgreePct>99.95") 